# 04 — Session d'entraînement (Colab T4)

**Objectif** : lancer une **vraie** session d'entraînement sur GPU, robuste aux coupures Colab grâce à la reprise automatique sur checkpoint.

**Si Colab coupe** : relancer toutes les cellules depuis le haut. La cellule de reprise détectera `last.pt` sur Drive et continuera là où ça s'est arrêté.

## 1. Setup Colab (Drive + repo + dépendances)

À ignorer si on tourne en local. Sinon, exécuter dans l'ordre.

In [ ]:
# Montage Drive (Colab uniquement)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab :", IN_COLAB)

In [ ]:
# Clone / pull du repo + cd dedans + install
import os, subprocess
REPO_DIR = "/content/Filtre-Voix-DL" if IN_COLAB else os.path.abspath("..")
REPO_URL = "https://github.com/Theo-Lempereur/Filtre-Voix-DL.git"
BRANCH   = "features/training"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "fetch"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run(["pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("REPO_DIR :", REPO_DIR)

In [ ]:
# Vérification GPU
import torch
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

## 2. Wandb (optionnel)

Si on veut un dashboard en ligne. Sinon, on met `use_wandb=False` plus bas — les logs JSONL locaux suffisent.

In [ ]:
# Décommenter et exécuter une seule fois pour s'authentifier :
# import wandb; wandb.login()

## 3. Configuration du run

Le `run_id` doit être **unique** : il sert de nom de dossier pour les checkpoints (`checkpoints/<run_id>/`) et les logs (`logs/<run_id>/`). Convention : `YYYYMMDD-<tag>`.

In [ ]:
from src import config as cfg
from datetime import datetime

RUN_TAG = "base32-lr1e3"             # <-- éditer pour chaque expérience
RUN_ID  = f"{datetime.now():%Y%m%d}-{RUN_TAG}"

run_config = {
    "run_id":               RUN_ID,
    "lr":                   1e-3,
    "weight_decay":         0.0,
    "batch_size":           8,
    "num_workers":          2,
    "num_epochs":           50,
    "base_channels":        32,
    "seed":                 42,
    "grad_clip_norm":       1.0,
    "early_stop_patience":  7,
    "lr_patience":          3,
    "keep_last_n_ckpt":     3,
    "ckpt_every_n_epochs":  1,
    "use_wandb":            True,    # mettre à False si pas de compte wandb
    "notes":                "Baseline U-Net base=32, MSE magnitude.",
}
print("RUN_ID :", RUN_ID)

## 4. Reprise automatique sur checkpoint

Si un `last.pt` existe déjà dans `checkpoints/<run_id>/`, on le détecte et on le passe à `train(resume_from=...)`. Sinon, on démarre à zéro.

**C'est cette cellule qui rend la session robuste aux coupures Colab.**

In [ ]:
from pathlib import Path
from src.checkpoint import find_latest_checkpoint

ckpt_dir = Path(cfg.CHECKPOINTS) / RUN_ID
resume_from = find_latest_checkpoint(ckpt_dir)

if resume_from is None:
    print(f"[reprise] aucun checkpoint dans {ckpt_dir} → entraînement neuf.")
else:
    print(f"[reprise] checkpoint détecté : {resume_from}")
    print("          l'entraînement reprendra à partir de cet état.")

## 5. Lancement de l'entraînement

In [ ]:
from src.train import train

history = train(run_config, resume_from=resume_from)

## 6. Vérification — les checkpoints sont bien sur Drive

In [ ]:
ckpt_dir = Path(cfg.CHECKPOINTS) / RUN_ID
log_path = Path(cfg.LOGS) / RUN_ID / "history.jsonl"

print("Checkpoints :")
for p in sorted(ckpt_dir.iterdir()):
    size_mb = p.stat().st_size / 1e6
    print(f"  {p.name:<20s}  {size_mb:6.1f} MB")

print(f"\nJSONL d'historique : {log_path}")
print("existe :", log_path.is_file())

## 7. Aperçu rapide

Quatre courbes pour avoir une idée immédiate. Pour l'analyse fine (gap, flags d'overfit, comparaison de runs) → notebook [05_training_analysis.ipynb](05_training_analysis.ipynb).

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
ep = history["epoch"]

axes[0, 0].plot(ep, history["train_loss"], label="train")
axes[0, 0].plot(ep, history["val_loss"],   label="val")
axes[0, 0].set_title("Loss MSE"); axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(ep, history["val_si_sdr"], color="tab:green")
axes[0, 1].set_title("SI-SDR validation (dB, plus haut = mieux)"); axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(ep, history["lr"], color="tab:orange")
axes[1, 0].set_yscale("log")
axes[1, 0].set_title("Learning rate"); axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(ep, history["gap"], color="tab:red")
axes[1, 1].set_title("Gap (val - train) — un gap qui augmente = overfit")
axes[1, 1].axhline(0, color="k", linestyle="--", linewidth=0.5)
axes[1, 1].grid(alpha=0.3)

for ax in axes.flatten():
    ax.set_xlabel("epoch")
fig.tight_layout()
plt.show()